# WESAD — Train Students via Knowledge Distillation (Session 2)

**Before running:**
1. Notebook Settings -> Accelerator -> **GPU T4 x2** (or P100).
2. Notebook Settings -> Internet -> **On**.
3. Add Data -> attach the same WESAD dataset used in Session 1.
4. Add Data -> attach the **teacher checkpoints** from Session 1 — either:
   - that notebook's own output (Add Data -> Your Work -> Notebook Output), or
   - a small dataset you made from the 15 downloaded `teacher_loso_S*.pt` files.

Trains all 4 student models (MicroCNN, TinyCNN, MiniCNN-LSTM, + ablation variant)
in both `standalone` and `distilled` modes, 15 LOSO folds each = 8 runs x 15 folds.

If this doesn't fit in one 12h session, use the `MODEL_FILTER` / `MODE_FILTER`
cell below to split the work across multiple runs (e.g. standalone first, then
distilled) — no code changes needed, `train_students.py` already supports this.

In [ ]:
import os, glob

# Auto-detect the attached WESAD dataset by locating the S2/S2.pkl marker
# file anywhere under /kaggle/input. Recursive so it doesn't matter how the
# dataset was packaged (some mirrors add an extra top-level 'WESAD' folder).
candidates = glob.glob('/kaggle/input/**/S2/S2.pkl', recursive=True)
if not candidates:
    print('No match. Top-level /kaggle/input contents:',
          os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else '(missing)')
assert candidates, (
    "WESAD dataset not found under /kaggle/input. "
    "Attach it via 'Add Data' first (must contain S2/S2.pkl ... S17/S17.pkl)."
)
data_root = os.path.dirname(os.path.dirname(candidates[0]))
print('Detected WESAD data root:', data_root)

os.environ['WESAD_DATA_DIR'] = data_root
os.environ['WESAD_OUTPUT_DIR'] = '/kaggle/working/outputs'

In [ ]:
REPO_DIR = '/kaggle/working/healthcare_wesad'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/RiverRover-stack/healthcare_wesad.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Bridge Session 1's teacher checkpoints into this session's MODELS_DIR.
# IMPORTANT: MODELS_DIR comes from src/config.py, which resolves it from the
# WESAD_OUTPUT_DIR env var set in cell 1 (/kaggle/working/outputs/models).
# Do NOT use a relative 'outputs/models' path here: after the %cd above, that
# resolves inside the cloned repo instead, the pipeline never looks there, and
# every distilled LOSO fold silently skips with "teacher checkpoint not found".
import shutil, sys
sys.path.insert(0, 'src')
from config import MODELS_DIR, REPORTS_DIR, create_directories

create_directories()
print('MODELS_DIR :', MODELS_DIR)
print('REPORTS_DIR:', REPORTS_DIR)

found = glob.glob('/kaggle/input/**/teacher_loso_S*.pt', recursive=True)
assert found, (
    'No teacher checkpoints found under /kaggle/input. '
    "Attach Session 1's output (or a dataset of the 15 teacher_loso_S*.pt files) via 'Add Data'."
)
for f in found:
    shutil.copy(f, MODELS_DIR)

copied = sorted(MODELS_DIR.glob('teacher_loso_S*.pt'))
print(f'Copied {len(copied)} teacher checkpoints into {MODELS_DIR}:')
for c in copied:
    print(' ', c.name)
assert len(copied) == 15, f'Expected 15 teacher checkpoints, found {len(copied)}.'


In [ ]:
# Sanity checks before committing to the full run.
import sys
sys.path.insert(0, 'src')

import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not enabled — check Notebook Settings -> Accelerator.'

from data import load_all_subjects
_probe = load_all_subjects(subject_ids=['S2'])
assert 'S2' in _probe, 'Failed to load S2 — check dataset path/structure.'
print('Sanity check passed.')

In [ ]:
# Edit these to split the work across sessions if 12h isn't enough for the
# full 8-run sweep. Examples:
#   MODE_FILTER = 'standalone'   # run standalone first, distilled in a later session
#   MODEL_FILTER = 'MicroCNN'    # one model at a time
MODEL_FILTER = 'all'
MODE_FILTER  = 'both'

args = []
if MODEL_FILTER != 'all':
    args += ['--model', MODEL_FILTER]
if MODE_FILTER != 'both':
    args += ['--mode', MODE_FILTER]

cmd = 'python train_students.py ' + ' '.join(args)
print('Running:', cmd)
!{cmd}

# `!` does not raise on a non-zero exit, so check explicitly. Without this the
# run fails here but you only find out two cells later as a FileNotFoundError.
assert _exit_code == 0, f'train_students.py failed with exit code {_exit_code} - see the log above.'


In [ ]:
# Show the resulting comparison table, then zip everything for download.
csv_path = REPORTS_DIR / 'model_comparison.csv'
assert csv_path.exists(), (
    f'{csv_path} was not written. The training cell above did not complete any '
    'LOSO fold. Scroll up: if every fold says "teacher checkpoint not found", '
    'the bridge cell put the checkpoints in the wrong directory.'
)
print(csv_path.read_text())

!cd /kaggle/working && zip -rq student_outputs.zip outputs/models outputs/reports
print('Saved /kaggle/working/student_outputs.zip - download it from the notebook Output tab.')
